# TimePatch-SSL: Reproducible Experimental Notebook

This notebook contains the original experimental workflow used to develop and evaluate the TimePatch-SSL framework for multimodal physiological stress detection.

## Scope

The workflow includes:

- Loading and preprocessing the WESAD dataset.
- Loading and preprocessing the Stress-Predict processed dataset.
- Resampling BVP, EDA, and TEMP signals to a common sampling rate.
- Binary stress-label construction and window generation.
- Standard and hierarchical Time-Patch Transformer models.
- Contrastive self-supervised learning with the NT-Xent objective.
- Subject-independent supervised and SSL experiments.
- Optuna/TPE hyperparameter optimization.
- Cross-dataset transfer experiments.
- Combined-dataset experiments.
- Subject-level Wilcoxon signed-rank statistical analysis.
- Additional subject-partition documentation.

> **Reproducibility note.** This organized version preserves the computational workflow of the original notebook. Comments and documentation have been rewritten in formal English. Dataset paths and experiment-output paths should be configured for the local execution environment before running the notebook.


## Notebook Organization

1. Global configuration and dataset paths
2. WESAD data loading and preprocessing
3. Stress-Predict / Stress-ID data loading and preprocessing
4. PyTorch datasets and SSL augmentations
5. Time-Patch Transformer architectures
6. Self-supervised and supervised training
7. Subject-independent evaluation
8. Hyperparameter optimization with Optuna/TPE
9. Cross-dataset transfer evaluation
10. Combined-dataset evaluation
11. Subject-level statistical analysis
12. Additional 80/20 subject-level evaluation
13. Partition and index documentation


## Participant-Level Leakage Prevention

All train, validation, and test partitions must be created from **participant identifiers before window indices are selected**. A participant must never occur in more than one partition. Cross-dataset target data must remain evaluation-only.

Normalization and missing-value imputation parameters must be estimated from the training partition only and then applied unchanged to validation and test data.


In [37]:
from sklearn.model_selection import train_test_split
import numpy as np

def assert_disjoint_participants(train_groups, val_groups=None, test_groups=None):
    """Verify that participant identities do not overlap across partitions."""
    train_ids = set(np.unique(train_groups).tolist())
    val_ids = set() if val_groups is None else set(np.unique(val_groups).tolist())
    test_ids = set() if test_groups is None else set(np.unique(test_groups).tolist())
    assert train_ids.isdisjoint(val_ids), f"Training/validation participant leakage: {train_ids & val_ids}"
    assert train_ids.isdisjoint(test_ids), f"Training/test participant leakage: {train_ids & test_ids}"
    assert val_ids.isdisjoint(test_ids), f"Validation/test participant leakage: {val_ids & test_ids}"
    return True

def subject_level_split_indices(groups, test_size=0.20, random_state=42):
    """Split participant IDs first and return the corresponding window indices."""
    groups = np.asarray(groups)
    participant_ids = np.unique(groups)
    train_ids, test_ids = train_test_split(
        participant_ids, test_size=test_size, random_state=random_state, shuffle=True
    )
    train_idx = np.flatnonzero(np.isin(groups, train_ids))
    test_idx = np.flatnonzero(np.isin(groups, test_ids))
    assert_disjoint_participants(groups[train_idx], test_groups=groups[test_idx])
    return train_idx, test_idx, train_ids, test_ids

def subject_level_train_val_indices(groups, val_size=0.15, random_state=42):
    """Create a participant-disjoint training/validation partition."""
    groups = np.asarray(groups)
    participant_ids = np.unique(groups)
    train_ids, val_ids = train_test_split(
        participant_ids, test_size=val_size, random_state=random_state, shuffle=True
    )
    train_idx = np.flatnonzero(np.isin(groups, train_ids))
    val_idx = np.flatnonzero(np.isin(groups, val_ids))
    assert_disjoint_participants(groups[train_idx], val_groups=groups[val_idx])
    return train_idx, val_idx, train_ids, val_ids

def make_subject_holdout_split(groups, train_fraction=0.80, random_state=42):
    """Create a participant-disjoint train/test split."""
    groups = np.asarray(groups)
    subjects = np.unique(groups)
    train_subjects, test_subjects = train_test_split(
        subjects, train_size=train_fraction, random_state=random_state, shuffle=True
    )
    train_idx = np.flatnonzero(np.isin(groups, train_subjects))
    test_idx = np.flatnonzero(np.isin(groups, test_subjects))
    overlap = set(train_subjects).intersection(set(test_subjects))
    if overlap:
        raise RuntimeError(f"Participant leakage detected: {sorted(overlap)}")
    print(f"Training participants: {len(train_subjects)} | Test participants: {len(test_subjects)} | Overlap: 0")
    return train_idx, test_idx, train_subjects, test_subjects



In [178]:

!pip -q install numpy scipy scikit-learn matplotlib torch tqdm pandas


In [38]:

import os
import pickle
import random
import numpy as np

from scipy import signal
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


DEVICE: cuda


### Training-Only Normalization and Imputation

The following transformer estimates channel-wise meanns and scales **only from training windows**. Validation and test data are transformed using those fixed training-derived parameters. This avoids evaluation-partition information entering preprocessing parameter estimation.


In [39]:
class TrainingOnlyRobustScaler:
    """Robust channel-wise preprocessing fitted exclusively on training data."""

    def __init__(self, eps=1e-6):
        self.eps = eps
        self.meann_ = None
        self.scale_ = None

    def fit(self, X_train):
        X_train = np.asarray(X_train, dtype=np.float32)
        if X_train.ndim != 3:
            raise ValueError("Expected shape: (n_windows, time, channels).")
        self.meann_ = np.nanmeann(X_train, axis=(0, 1)).astype(np.float32)
        q75 = np.nanpercentile(X_train, 75, axis=(0, 1))
        q25 = np.nanpercentile(X_train, 25, axis=(0, 1))
        iqr = q75 - q25
        std = np.nanstd(X_train, axis=(0, 1))
        self.scale_ = np.where(iqr > self.eps, iqr, std + self.eps).astype(np.float32)
        return self

    def transform(self, X):
        if self.meann_ is None:
            raise RuntimeError("Fit the scaler on training data before transformation.")
        X = np.asarray(X, dtype=np.float32).copy()
        for ch in range(X.shape[-1]):
            invalid = ~np.isfinite(X[..., ch])
            X[..., ch][invalid] = self.meann_[ch]
        return ((X - self.meann_) / self.scale_).astype(np.float32)

    def fit_transform(self, X_train):
        return self.fit(X_train).transform(X_train)

def preprocess_train_val_test(X_train, X_val=None, X_test=None):
    """Fit preprocessing on training data and reuse it unchanged elsewhere."""
    scaler = TrainingOnlyRobustScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = None if X_val is None else scaler.transform(X_val)
    X_test = None if X_test is None else scaler.transform(X_test)
    return X_train, X_val, X_test, scaler


## 1. Global Configuration and Dataset Paths

Expected WESAD directory structure:

```text
WESAD/
├── S2/
│   └── S2.pkl
├── S3/
│   └── S3.pkl
└── ...
```

Update the dataset and results paths in the following cell before execution.


In [40]:
WESAD_ROOT = r"C:\Users\USUARIO\Desktop\TimePatch-SSL\data\WESAD\WESAD"   # Update this path for the local dataset location.
RESULTS_DIR = r"C:\Users\USUARIO\Desktop\TimePatch-SSL\results_optuna\results_optuna"

# Target sampling frequency.
TARGET_FS = 4
# Window duration in seconds.
WINDOW_SECONDS = 30
# Window stride in seconds.
WINDOW_STRIDE_SECONDS = 15

USE_LABELS = {1, 2, 3}   # WESAD labels: baseline, stress, amusement
STRESS_LABEL = 2
NOSTRESS_LABELS = {1, 3} # Binary non-stress labels.

# Normalization method: "median" (recommended, robust to physiological spikes) or "mean"
NORM_METHOD = "median"

# Model architecture flag:
# USE_TIMEMAE = False  -> Modelo Base con PatchEmbed1D (sin TimeMAE, recomendado)
# USE_TIMEMAE = True   -> Modelo con TimeMAEPatchEmbed (con TimeMAE)
USE_TIMEMAE = True


## 2. WESAD Data Loading and Preprocessing

This section loads wrist BVP, EDA, and TEMP signals, resamples them to the common target sampling rate, applies the preprocessing implemented in the original workflow, and generates binary stress-detection windows.


In [41]:

# WESAD signals may be recorded at different sampling frequencies.
# Resample each signal to the common target sampling frequency.
#all modalities are converted to 4 Hz
# This ensures temporal alignment across the three modalities before model input.
def resample_1d(x, orig_fs, target_fs):
    x = np.asarray(x).squeeze()
    if orig_fs == target_fs:
        return x.astype(np.float32)
    duration = len(x) / float(orig_fs)
    target_len = max(2, int(round(duration * target_fs)))
    return signal.resample(x, target_len).astype(np.float32)

# Apply robust normalization to each signal using:
# - meann
#interquartile range

# Robust scaling is used because physiological signals may contain:
#picos
#ruido
# outliers

# This normalization reduces scale differences across recordings.
def robust_norm(x, method=None):
    if method is None:
        method = globals().get("NORM_METHOD", "median")
    x = np.asarray(x, dtype=np.float32)
    if np.isnan(x).any():
        nan_val = np.nanmedian(x) if method == "median" else np.nanmean(x)
        x = np.nan_to_num(x, nan=nan_val if np.isnan(x).any() else 0.0)
    med = np.median(x) if method == "median" else np.mean(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    scale = iqr if iqr > 1e-6 else (np.std(x) + 1e-6)
    return ((x - med) / scale).astype(np.float32)

# Load each participant pickle file and extract only:

#BVP
#EDA
#TEMP
#label

#Luego:

# Resample each modality to 4 Hz.
# Align signals and labels.
#normaliza
# Construct a three-channel temporal array.
def load_wesad_subject(pkl_path):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    wrist = data["signal"]["wrist"]
    labels = np.asarray(data["label"]).astype(int).squeeze()

    bvp = np.asarray(wrist["BVP"]).squeeze()
    eda = np.asarray(wrist["EDA"]).squeeze()
    temp = np.asarray(wrist["TEMP"]).squeeze()

    fs_bvp = 64
    fs_eda = 4
    fs_temp = 4
    fs_label = 700

    bvp = resample_1d(bvp, fs_bvp, TARGET_FS)
    eda = resample_1d(eda, fs_eda, TARGET_FS)
    temp = resample_1d(temp, fs_temp, TARGET_FS)
    labels = resample_1d(labels, fs_label, TARGET_FS).round().astype(int)

    L = min(len(bvp), len(eda), len(temp), len(labels))
    X = np.stack([
        robust_norm(bvp[:L]),
        robust_norm(eda[:L]),
        robust_norm(temp[:L]),
    ], axis=1)

    labels = labels[:L]
    keep = np.isin(labels, list(USE_LABELS))
    X = X[keep]
    labels = labels[keep]
    y = np.where(labels == STRESS_LABEL, 1, 0).astype(np.int64)
    return X, y

# Segment the continuous recording into windows.

# Window duration is controlled by WINDOW_SECONDS.
#con stride de 30 s

# Windowing converts each continuous participant recording into model-ready samples.

# Each window has the following approximate shape:

#tiempo x canales
#o sea: T x 3
def segment_windows(X, y, window_seconds=60, stride_seconds=30, fs=4):
    win = int(window_seconds * fs)
    stride = int(stride_seconds * fs)
    Xw, yw = [], []

    for start in range(0, len(X) - win + 1, stride):
        end = start + win
        xw = X[start:end]
        yw_raw = y[start:end]
        frac = yw_raw.mean()
        if frac in [0.0, 1.0] or frac <= 0.2 or frac >= 0.8:
            Xw.append(xw.astype(np.float32))
            yw.append(int(frac >= 0.5))

    if len(Xw) == 0:
        return np.zeros((0, win, X.shape[1]), dtype=np.float32), np.zeros((0,), dtype=np.int64)

    return np.stack(Xw), np.asarray(yw, dtype=np.int64)

# Combine windows from all participants into the arrays below.

#X: windows
# y: binary labels.
# groups: participant identifier.

# `groups` stores participant identity for subject-independent evaluation.

def load_all_wesad(root):
    X_all, y_all, groups_all = [], [], []
    subject_dirs = sorted([d for d in os.listdir(root) if d.startswith("S")])

    for subj in subject_dirs:
        pkl_path = os.path.join(root, subj, f"{subj}.pkl")
        if not os.path.exists(pkl_path):
            print("Skipping:", pkl_path)
            continue
        X, y = load_wesad_subject(pkl_path)
        Xw, yw = segment_windows(X, y, WINDOW_SECONDS, WINDOW_STRIDE_SECONDS, TARGET_FS)
        if len(Xw) == 0:
            continue
        sid = int(subj[1:])
        X_all.append(Xw)
        y_all.append(yw)
        groups_all.append(np.full(len(yw), sid))
        print(subj, Xw.shape, "stress ratio=", float(yw.mean()))

    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    groups_all = np.concatenate(groups_all, axis=0)
    return X_all, y_all, groups_all

X, y, groups = load_all_wesad(WESAD_ROOT)
print("X:", X.shape, "y:", y.shape, "groups:", groups.shape)
print("Subjects:", np.unique(groups))
print("Stress ratio:", float(y.mean()))


S10 (148, 120, 3) stress ratio= 0.3108108108108108
S11 (145, 120, 3) stress ratio= 0.30344827586206896
S13 (146, 120, 3) stress ratio= 0.2945205479452055
S14 (145, 120, 3) stress ratio= 0.30344827586206896
S15 (147, 120, 3) stress ratio= 0.30612244897959184
S16 (144, 120, 3) stress ratio= 0.2986111111111111
S17 (149, 120, 3) stress ratio= 0.3087248322147651
S2 (138, 120, 3) stress ratio= 0.2898550724637681
S3 (140, 120, 3) stress ratio= 0.3
S4 (142, 120, 3) stress ratio= 0.2887323943661972
S5 (145, 120, 3) stress ratio= 0.2827586206896552
S6 (143, 120, 3) stress ratio= 0.2937062937062937
S7 (144, 120, 3) stress ratio= 0.2847222222222222
S8 (145, 120, 3) stress ratio= 0.296551724137931
S9 (143, 120, 3) stress ratio= 0.2937062937062937
X: (2164, 120, 3) y: (2164,) groups: (2164,)
Subjects: [ 2  3  4  5  6  7  8  9 10 11 13 14 15 16 17]
Stress ratio: 0.29713493530499074


## 3. Stress-Predict / Stress-ID Data Loading and Preprocessing

This section loads the processed BVP, EDA, and TEMP signals for the second dataset and converts them to the representation expected by the Time-Patch Transformer pipeline.


In [183]:
# Loader for processed BVP, EDA, and TEMP data used by the Time-Patch Transformer.
import pandas as pd
import numpy as np

def robust_norm(x, method=None):
    if method is None:
        method = globals().get("NORM_METHOD", "median")
    x = np.asarray(x, dtype=np.float32)
    if np.isnan(x).any():
        nan_val = np.nanmedian(x) if method == "median" else np.nanmean(x)
        x = np.nan_to_num(x, nan=nan_val if np.isnan(x).any() else 0.0)
    med = np.median(x) if method == "median" else np.mean(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    scale = iqr if iqr > 1e-6 else (np.std(x) + 1e-6)
    return ((x - med) / scale).astype(np.float32)

def resample_1d(signal, target_len):
    signal = np.asarray(signal, dtype=np.float32)
    if len(signal) == target_len:
        return signal
    if len(signal) < 2:
        return np.full(target_len, float(signal[0]) if len(signal) else 0.0, dtype=np.float32)
    xp = np.linspace(0, 1, len(signal))
    x = np.linspace(0, 1, target_len)
    return np.interp(x, xp, signal).astype(np.float32)

def build_dataset_from_processed(sensor_files, label_col="TASK_LABEL", target_length=int(WINDOW_SECONDS * TARGET_FS)):
    # Metadata columns, including Participant, are excluded from signal features.
    meta_cols = {"Participant","Activity","PSS_BEFORE","PSS_AFTER","BASELINE_LABEL","TASK_LABEL","axis"}
    frames = {}

    for name, path in sensor_files.items():
        df = pd.read_csv(path, index_col=0)
        # Construct a clean `Participant` column from the index and avoid duplicates.
        participants_index = df.index.astype(str)
        df = df.reset_index(drop=True)
        df["Participant"] = participants_index

        # Comprobar existencia de label_col
        if label_col not in df.columns:
            raise ValueError(f"label_col '{label_col}' no está en {path}; columnas: {list(df.columns)[:10]}")

        # Signal columns include numeric variables excluding metadata columns.
        sig_cols = [c for c in df.columns if c not in meta_cols and c not in ("Participant","Activity")]
        
        # Participant-level normalization.
        norm_method = globals().get("NORM_METHOD", "median")
        for part, group in df.groupby("Participant"):
            flat_sig = group[sig_cols].values.astype(np.float32).flatten()
            if np.isnan(flat_sig).any():
                nan_val = np.nanmedian(flat_sig) if norm_method == "median" else np.nanmean(flat_sig)
                flat_sig = np.nan_to_num(flat_sig, nan=nan_val if np.isnan(flat_sig).any() else 0.0)
            med = np.median(flat_sig) if norm_method == "median" else np.mean(flat_sig)
            iqr = np.percentile(flat_sig, 75) - np.percentile(flat_sig, 25)
            scale = iqr if iqr > 1e-6 else (np.std(flat_sig) + 1e-6)
            
            norm_vals = (df.loc[group.index, sig_cols].values.astype(np.float32) - med) / scale
            df.loc[group.index, sig_cols] = norm_vals
        # -----------------------------------------------

        frames[name] = df[["Participant","Activity"] + sig_cols + [label_col]].copy()

    # Add a window index by Participant and Activity to align modalities.
    for k, df in frames.items():
        df["window_idx"] = df.groupby(["Participant","Activity"]).cumcount()
        frames[k] = df

    # Construct the merge key: Participant + Activity + window_idx.
    for k in frames:
        frames[k]["_key"] = frames[k]["Participant"].astype(str) + "||" + frames[k]["Activity"].astype(str) + "||" + frames[k]["window_idx"].astype(str)

    # Retain keys shared across sensor modalities.
    common_keys = set.intersection(*[set(frames[k]["_key"].values) for k in frames])
    common_keys = sorted(common_keys)

    X_list = []
    y_list = []
    participants_list = []

    for key in common_keys:
        channel_data = []
        label_val = None
        part = None
        skip = False

        for name, df in frames.items():
            row = df[df["_key"] == key]
            if row.shape[0] != 1:
                skip = True
                break
            row = row.iloc[0]
            # Select the signal columns stored in frames[name].
            sig_cols = [c for c in df.columns if c not in meta_cols and c not in ("Participant","Activity","window_idx","_key",label_col)]
            sig = row[sig_cols].values.astype(np.float32)
            sig = resample_1d(sig, target_length)
            channel_data.append(sig)

            if label_val is None:
                # Etiquetar basado en Activity en lugar de PSS_AFTER
                activity = row['Activity']
                if activity == 'Baseline':
                    label_val = 0
                else:  # Stroop o Interview
                    label_val = 1
            if part is None:
                part = row["Participant"]

        if skip:
            continue

        X_list.append(np.stack(channel_data, axis=1))
        y_list.append(label_val)
        participants_list.append(part)

    if len(X_list) == 0:
        return np.zeros((0, target_length, len(frames),), dtype=np.float32), np.zeros((0,), dtype=np.int64), np.zeros((0,), dtype=np.int64), []

    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype=np.int64)
    groups = np.array([int(p.lstrip("S")) for p in participants_list], dtype=np.int64)

    return X, y, groups, participants_list

sensor_files = {
    "BVP": f"/home/leoisidro/CICLOS/X/PFC_II/PFC2/processed_data/BVP_{WINDOW_SECONDS}_seg.csv",
    "EDA": f"/home/leoisidro/CICLOS/X/PFC_II/PFC2/processed_data/EDA_{WINDOW_SECONDS}_seg.csv",
    "TEMP": f"/home/leoisidro/CICLOS/X/PFC_II/PFC2/processed_data/TEMP_{WINDOW_SECONDS}_seg.csv",
}

X_proc, y_proc, groups_proc, participants = build_dataset_from_processed(sensor_files, label_col="TASK_LABEL", target_length=int(WINDOW_SECONDS * TARGET_FS))
print("Loaded processed_data -> X.shape:", X_proc.shape, "y.shape:", y_proc.shape, "groups.shape:", groups_proc.shape)

X1, y1, groups_1 = X_proc, y_proc, groups_proc


Loaded processed_data -> X.shape: (1616, 240, 3) y.shape: (1616,) groups.shape: (1616,)


## 4. PyTorch Datasets and SSL Augmentations

This section defines dataset wrappers and stochastic physiological-signal augmentations used to construct positive pairs for contrastive self-supervised learning.


In [42]:

# Convert X and y to tensors compatible with the PyTorch DataLoader.

# This wrapper is used for:

# - supervised training
# - self-supervised training
#validation
class WindowDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]


# Create a stochastic augmented view of an input window.
# The augmentation pipeline applies small perturbations such as:

# Add low-amplitude noise.
#escalar amplitud
# Apply a small temporal shift.
# Mask a short segment of one channel.

# SSL uses two independently augmented views of the same sample as a positive pair.
def weak_augment(x):
    x = x.clone()
    if random.random() < 0.8:
        x = x + 0.02 * torch.randn_like(x)
    if random.random() < 0.5:
        x = x * torch.empty((1, x.shape[1])).uniform_(0.9, 1.1)
    if random.random() < 0.4:
        shift = random.randint(-5, 5)
        x = torch.roll(x, shifts=shift, dims=0)
    if random.random() < 0.3:
        c = random.randrange(x.shape[1])
        start = random.randrange(0, max(1, x.shape[0] - 10))
        width = random.randrange(5, 15)
        x[start:start+width, c] = 0
    return x

# Generate augmented pairs (x1, x2), where:

# x1 is one augmented view of the window.
# x2 is an independently augmented view of the same window.

# These transformations are used for contrastive learning.
class SSLPairDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        return weak_augment(x), weak_augment(x)


## 5. Time-Patch Transformer Architectures

This section defines the temporal patch representation and Transformer-based encoders used in the experiments, including the standard and hierarchical patch-encoding variants.


In [43]:
import sys
import os

# Ensure that the TimeMAE dependency is available in the execution environment.
if not os.path.exists('TimeMAE'):
    os.system('git clone https://github.com/ustc-time-series/TimeMAE.git')

if 'TimeMAE' not in sys.path:
    sys.path.append('TimeMAE')

import torch
import torch.nn as nn
from TimeMAE.model.TimeMAE import Encoder
from TimeMAE.model.layers import PositionalEmbedding

class TimeMAEPatchEmbed(nn.Module):
    def __init__(self, in_ch=3, patch_len=8, emb_dim=128, max_patches=5000):
        super().__init__()
        self.patch_len = patch_len
        self.input_projection = nn.Conv1d(in_ch, emb_dim, kernel_size=patch_len, stride=patch_len)
        self.position = PositionalEmbedding(max_patches, emb_dim)
        
        class Args: pass
        args = Args()
        args.d_model = emb_dim
        args.attn_heads = 4
        args.layers = 2
        args.dropout = 0.1
        args.enable_res_parameter = 1
        self.encoder = Encoder(args)

    def forward(self, x):
        B, T, C = x.shape
        P = self.patch_len
        T2 = (T // P) * P
        x = x[:, :T2, :]
        x = self.input_projection(x.transpose(1, 2)).transpose(1, 2).contiguous()
        pos_emb = self.position(x)
        x += pos_emb[:, :x.size(1), :]
        x = self.encoder(x)
        return x

# Divide the sequence into short temporal blocks called patches.

# Example:

#si patch_len = 8
# Consecutive temporal samples are grouped into patches.

# Each patch is mapped to a latent representation.

# This reduces the effective sequence length processed by the Transformer.

# Reduce the effective sequence length.
# This produces the token representation consumed by the Transformer.
#captura patrones locales antes de modelar dependencias largas
class PatchEmbed1D(nn.Module):
    def __init__(self, in_ch=3, patch_len=8, emb_dim=128):
        super().__init__()
        self.patch_len = patch_len
        self.proj = nn.Linear(in_ch * patch_len, emb_dim)

    def forward(self, x):
        B, T, C = x.shape
        P = self.patch_len
        T2 = (T // P) * P
        x = x[:, :T2, :].reshape(B, T2 // P, P * C)
        return self.proj(x)

# After patch construction, the model adds:

#un CLS token
#embeddings posicionales
#un TransformerEncoder

# The CLS token is subsequently used as the global window representation.

# The model supports representations used for downstream classification and SSL.

# Representation used for SSL.
# Binary stress/non-stress classification.

# The Transformer therefore learns a multimodal temporal representation of each window.
class TimePatchTransformer(nn.Module):
    def __init__(self, in_ch=3, patch_len=8, emb_dim=128, depth=4, heads=4, dropout=0.1, num_classes=2, use_timemae=None):
        super().__init__()
        if use_timemae is None:
            use_timemae = globals().get("USE_TIMEMAE", False)
        self.use_timemae = use_timemae

        if self.use_timemae:
            self.patch = TimeMAEPatchEmbed(in_ch, patch_len, emb_dim)
        else:
            self.patch = PatchEmbed1D(in_ch, patch_len, emb_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, emb_dim))
        self.pos_emb = nn.Parameter(torch.zeros(1, 512, emb_dim))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm = nn.LayerNorm(emb_dim)
        self.ssl_head = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, 64)
        )
        self.cls_head = nn.Linear(emb_dim, num_classes)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

    def forward_features(self, x):
        x = self.patch(x)
        B, N, D = x.shape
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_emb[:, :N+1, :]
        x = self.encoder(x)
        x = self.norm(x)
        return x[:, 0]

    def forward_ssl(self, x):
        z = self.forward_features(x)
        z = self.ssl_head(z)
        return F.normalize(z, dim=-1)

    def forward(self, x):
        z = self.forward_features(x)
        return self.cls_head(z)

print(f"TimePatchTransformer listo. Modo: {'Con TimeMAE (TimeMAEPatchEmbed)' if globals().get('USE_TIMEMAE', False) else 'Base / Sin TimeMAE (PatchEmbed1D)'}")


TimePatchTransformer listo. Modo: Con TimeMAE (TimeMAEPatchEmbed)


## 6. Self-Supervised and Supervised Training

This section implements NT-Xent contrastive learning, SSL pretraining, supervised fine-tuning, and model evaluation.


In [44]:
# NT-Xent contrastive loss.

# Objective:

# - augmented views of the same sample are treated as a positive pair;
# - views from different samples act as negatives.

# This objective learns representations without using class labels during SSL pretraining.
def nt_xent(z1, z2, temp=0.2):
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=-1) / temp
    sim.fill_diagonal_(-1e9)
    targets = torch.arange(B, device=z.device)
    targets = torch.cat([targets + B, targets], dim=0)
    return F.cross_entropy(sim, targets)

# Perform self-supervised pretraining.
#Pasos:

# Use training windows without class labels.
# Generate two independently augmented views.
# Encode both views.
# Compute the contrastive objective.
# Update model parameters.

# The resulting encoder parameters provide the initialization used for supervised fine-tuning.

def pretrain_ssl(model, X_unlabeled, epochs=6, batch_size=64, lr=1e-3):
    ds = SSLPairDataset(X_unlabeled)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=True)
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    model.train()
    for ep in range(epochs):
        losses = []
        for x1, x2 in dl:
            x1, x2 = x1.to(DEVICE), x2.to(DEVICE)
            z1 = model.forward_ssl(x1)
            z2 = model.forward_ssl(x2)
            loss = nt_xent(z1, z2)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        print(f"SSL epoch {ep+1}/{epochs}: {np.mean(losses):.4f}")
    return model

def evaluate_probs(y_true, probs):
    pred = (probs >= 0.5).astype(int)
    out = {
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
    }
    try:
        out["auroc"] = roc_auc_score(y_true, probs)
    except Exception:
        out["auroc"] = np.nan
    return out

def finetune(model, X_train, y_train, X_val, y_val, epochs=8, batch_size=64, lr=2e-4):
    train_dl = DataLoader(WindowDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(WindowDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_state = None
    best_f1 = -1

    for ep in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

        model.eval()
        probs_all, ys_all = [], []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(DEVICE)
                probs = torch.softmax(model(xb), dim=-1)[:, 1].cpu().numpy()
                probs_all.append(probs)
                ys_all.append(yb.numpy())

        probs_all = np.concatenate(probs_all)
        ys_all = np.concatenate(ys_all)
        metrics = evaluate_probs(ys_all, probs_all)
        print(f"FT epoch {ep+1}/{epochs}: loss={np.mean(train_losses):.4f}, val_f1={metrics['f1']:.4f}")

        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model

def predict_probs(model, X_test, batch_size=128):
    dl = DataLoader(WindowDataset(X_test), batch_size=batch_size, shuffle=False)
    model.eval()
    probs = []
    with torch.no_grad():
        for xb in dl:
            xb = xb.to(DEVICE)
            p = torch.softmax(model(xb), dim=-1)[:, 1].cpu().numpy()
            probs.append(p)
    return np.concatenate(probs)


## 7. Subject-Independent Evaluation

Leave-One-Group-Out (LOGO) evaluation is performed with participant identity as the grouping variable. Each held-out participant is used only for evaluation in the corresponding fold.


In [45]:
# Leave-One-Group-Out evaluation uses participant identity as the grouping variable.

# In each fold:

# - one complete participant is held out for testing;
# - the remaining participants are used for model development.

# This prevents windows from the same participant from appearing in both training and test partitions.

# Within each fold:
# (a) Separate development and test data.
# X_train and y_train contain all participants except the held-out participant.
# X_test and y_test contain the held-out participant.

# (b) Create a validation subset from the development data.
# A subset of the development data is used for validation.

# (c) Optional synthetic-data augmentation.
# If use_synth=True:
# Generate synthetic windows.
# Add the selected samples to the training set.

# (d) Optional SSL pretraining.
# If use_ssl=True:
# First perform SSL pretraining using training windows.
# Then perform supervised fine-tuning.

# (e) Evaluation.
# Finally, evaluate performance on the held-out participant.
# Report Accuracy, F1, Balanced Accuracy, and AUROC.
#accuracy
#F1
#balanced accuracy
#AUROC

def run_loso(X, y, groups, use_ssl=False, use_synth=False, ssl_epochs=6, ft_epochs=8, synth_ratio=0.5, emb_dim=128,depth=4,heads=4,dropout=0.1):
    logo = LeaveOneGroupOut()
    results = []

    for fold, (tr_idx, te_idx) in enumerate(logo.split(X, y, groups), start=1):
        held_out = int(np.unique(groups[te_idx])[0])
        X_train, y_train = X[tr_idx], y[tr_idx]
        X_test, y_test = X[te_idx], y[te_idx]

        # Participant-disjoint validation partition from development participants
        dev_subjs = np.unique(groups[tr_idx])
        if len(dev_subjs) > 1:
            tr_subjs, val_subjs = train_test_split(dev_subjs, test_size=0.15, random_state=SEED, shuffle=True)
            tr2_idx = np.flatnonzero(np.isin(groups[tr_idx], tr_subjs))
            val_idx = np.flatnonzero(np.isin(groups[tr_idx], val_subjs))
        else:
            n_val = max(1, int(0.15 * len(X_train)))
            val_idx = np.arange(n_val)
            tr2_idx = np.arange(n_val, len(X_train))

        X_tr, y_tr = X_train[tr2_idx], y_train[tr2_idx]
        X_val, y_val = X_train[val_idx], y_train[val_idx]

        if use_synth:
            Xs, ys = generate_synthetic_dataset(int(len(X_tr) * synth_ratio), class_balance=float(y_tr.mean()))
            X_tr = np.concatenate([X_tr, Xs], axis=0)
            y_tr = np.concatenate([y_tr, ys], axis=0)

        model = TimePatchTransformer(in_ch=3, patch_len=8, emb_dim=emb_dim, depth=depth, heads=heads,dropout=dropout, num_classes=2)

        if use_ssl:
            X_unlab = X_train.copy()
            if use_synth:
                Xu, _ = generate_synthetic_dataset(int(0.5 * len(X_train)), class_balance=float(y_train.mean()))
                X_unlab = np.concatenate([X_unlab, Xu], axis=0)
            model = pretrain_ssl(model, X_unlab, epochs=ssl_epochs)

        model = finetune(model, X_tr, y_tr, X_val, y_val, epochs=ft_epochs)
        probs = predict_probs(model, X_test)
        metrics = evaluate_probs(y_test, probs)
        metrics["subject"] = held_out
        results.append(metrics)
        print("Fold", fold, "subject", held_out, metrics)

    return results

from scipy.stats import t as _t_dist

def summarize(name, results, txt_path="", append=True, alpha=0.05):
    """Summarize results using the mean, standard deviation, and 95% confidence interval.

    Parameters
    ----------
    name : str - Name of the experiment.
    results : list of dict - List of result dictionaries. Each dictionary must contain the keys 'f1', 'balanced_accuracy', 'auroc', and 'accuracy'.
    txt_path : str, optional - Path to the text file in which the summarized results are saved.
    append : bool - If True, append the results to the output file. If False, overwrite the existing file.
    alpha : float, default=0.05 - Significance level used to compute the confidence interval. The default value of 0.05 corresponds to a 95% confidence interval.

    Returns
    -------
    dict with { metrics: { mean, std, ci95_low, ci95_high, n } }
    """
    lines = ["", name]
    print("\n" + "="*70)
    print(name)
    print("="*70)
    summary = {}

    for key in ["accuracy", "f1", "balanced_accuracy", "auroc"]:
        vals = np.array([r[key] for r in results], dtype=float)
        vals = vals[~np.isnan(vals)]
        n        = len(vals)
        mean_val = float(np.mean(vals))
        std_val  = float(np.std(vals, ddof=1))          # std muestral
        sem      = std_val / np.sqrt(n)
        t_crit   = float(_t_dist.ppf(1 - alpha / 2, df=max(n - 1, 1)))
        ci_half  = t_crit * sem
        summary[key] = {
            "mean":      mean_val,
            "std":       std_val,
            "ci95_low":  float(mean_val - ci_half),
            "ci95_high": float(mean_val + ci_half),
            "n":         n,
        }
        line = (f"  {key:22s}: {mean_val:.4f} ± {std_val:.4f}"
                f"  [IC95%: ({mean_val - ci_half:.4f}, {mean_val + ci_half:.4f})]  n={n}")
        lines.append(line)
        print(line)

    mode = "a" if append else "w"
    if txt_path:
        with open(txt_path, mode, encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")

    return summary


### 7.1 Supervised and SSL Evaluation Scenarios

Two principal training scenarios are evaluated:

1. **Supervised initialization:** the model is trained directly for binary stress classification.
2. **SSL initialization followed by supervised fine-tuning:** the encoder is first pretrained using contrastive learning and subsequently fine-tuned using labeled training data.


### 7.2 Initial WESAD Experiments



# 1) Supervised only
res_sup = run_loso(X, y, groups, use_ssl=False, use_synth=False, ft_epochs=8)
summarize("Supervised only", res_sup)



# 2) SSL + supervised
res_ssl = run_loso(X, y, groups, use_ssl=True, use_synth=False, ssl_epochs=6, ft_epochs=8)
summarize("SSL + supervised", res_ssl)


### 7.3 Initial Stress-Predict / Stress-ID Experiments


# 1) Supervised only
res_sup_1 = run_loso(X1, y1, groups_1, use_ssl=False, use_synth=False, ft_epochs=8)
summarize("Supervised only", res_sup_1)


res_ssl_1 = run_loso(X1, y1, groups_1, use_ssl=True, use_synth=False, ssl_epochs=6, ft_epochs=8)
summarize("SSL + supervised", res_ssl_1)


## 8. Hyperparameter Optimization with Optuna/TPE

This section performs hyperparameter optimization using Optuna. The selected configurations should be exported and reported alongside the search space for complete reproducibility.


import os
import json
from datetime import datetime

import numpy as np
import torch
import optuna


def objective_optuna(trial, X, y, groups):
    params = {
        'patch_len': trial.suggest_categorical('patch_len', [8, 16]),
        'emb_dim': trial.suggest_categorical('emb_dim', [128, 256]),
        'depth': trial.suggest_int('depth', 3, 6),
        'dropout': trial.suggest_float('dropout', 0.02, 0.25),
        'ft_epochs': trial.suggest_int('ft_epochs', 6, 10),
        'ssl_epochs': trial.suggest_int('ssl_epochs', 4, 10),
        'batch_size': trial.suggest_categorical('batch_size', [16, 32]),
        'use_ssl': trial.suggest_categorical('use_ssl', [True]),
    }

    torch.cuda.empty_cache()
    results = run_loso(
        X, y, groups,
        use_ssl=params['use_ssl'],
        ssl_epochs=params['ssl_epochs'],
        ft_epochs=params['ft_epochs'],
        synth_ratio=0.5,
        emb_dim=params['emb_dim'],
        depth=params['depth'],
        dropout=params['dropout'],
    )

    mean_f1 = float(np.nanmean([r['f1'] for r in results]))
    trial.set_user_attr('mean_accuracy', float(np.nanmean([r['accuracy'] for r in results])))
    trial.set_user_attr('mean_auroc', float(np.nanmean([r.get('auroc', np.nan) for r in results])))
    trial.set_user_attr('params', params)
    return mean_f1


def run_optuna_search(X, y, groups, n_trials=20, study_name='wesad_optuna'):
    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction='maximize', sampler=sampler, study_name=study_name)

    def _objective(trial):
        return objective_optuna(trial, X, y, groups)

    study.optimize(_objective, n_trials=n_trials)

    print('\n' + '=' * 80)
    print('OPTUNA COMPLETED')
    print('=' * 80)
    print(f'Best F1: {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')

    best_path = os.path.join(RESULTS_DIR, f'{study_name}.json')
    payload = {
        'name': study_name,
        'best_value': float(study.best_value),
        'best_params': study.best_params,
        'trials': [
            {
                'number': t.number,
                'value': None if t.value is None else float(t.value),
                'params': t.params,
                'user_attrs': t.user_attrs,
                'state': str(t.state),
            }
            for t in study.trials
        ],
    }
    with open(best_path, 'w') as f:
        json.dump(payload, f, indent=2)

    print(f'Results saved to: {best_path}')
    return study


print('Optuna listo')


### 8.1 Selected Hyperparameters for WESAD


# Run the Optuna hyperparameter search.
print("Starting OPTUNA...")
study_optuna = run_optuna_search(
    X, y, groups,
    n_trials=5,
    study_name=f"wesad_optuna_{WINDOW_SECONDS}seg"
)

print("\nMejor F1:", study_optuna.best_value)
print("Selected hyperparameters:", study_optuna.best_params)


In [46]:
# 1) Supervised only

import json

path = rf"C:\Users\USUARIO\Desktop\TimePatch-SSL\results_optuna\results_optuna\wesad_optuna_{WINDOW_SECONDS}seg.json"
#path = f"/home/leoisidro/CICLOS/X/PFC_II/PFC2/results_optuna/wesad_optuna_{WINDOW_SECONDS}seg.json"

with open(path, "r") as f:
    data = json.load(f)

best_params = data["best_params"]
print(best_params)

res_sup = run_loso(
X, y, groups,
use_ssl=False,
ssl_epochs=best_params["ssl_epochs"],
ft_epochs=best_params["ft_epochs"],
emb_dim=best_params["emb_dim"],
depth=best_params["depth"],
dropout=best_params["dropout"]
)
vartime = "TimeMAE" if USE_TIMEMAE else "Base"
summarize("Supervised only", res_sup,f"resultados_Supervisado_WESAD_{WINDOW_SECONDS}seg_{vartime}.txt")


{'patch_len': 8, 'emb_dim': 256, 'depth': 6, 'dropout': 0.20593139006678607, 'ft_epochs': 7, 'ssl_epochs': 4, 'batch_size': 16, 'use_ssl': True}
FT epoch 1/7: loss=0.3056, val_f1=0.8078
FT epoch 2/7: loss=0.2471, val_f1=0.8091
FT epoch 3/7: loss=0.2366, val_f1=0.8435
FT epoch 4/7: loss=0.2006, val_f1=0.8889
FT epoch 5/7: loss=0.1838, val_f1=0.8939
FT epoch 6/7: loss=0.1697, val_f1=0.7898
FT epoch 7/7: loss=0.1673, val_f1=0.8105
Fold 1 subject 2 {'accuracy': 1.0, 'f1': 1.0, 'balanced_accuracy': 1.0, 'auroc': 1.0, 'subject': 2}
FT epoch 1/7: loss=0.3138, val_f1=0.9236
FT epoch 2/7: loss=0.2410, val_f1=0.9373
FT epoch 3/7: loss=0.2190, val_f1=0.9110
FT epoch 4/7: loss=0.2088, val_f1=0.9110
FT epoch 5/7: loss=0.2008, val_f1=0.9275
FT epoch 6/7: loss=0.1932, val_f1=0.8821
FT epoch 7/7: loss=0.1590, val_f1=0.8759
Fold 2 subject 3 {'accuracy': 0.75, 'f1': 0.6728971962616822, 'balanced_accuracy': 0.7806122448979591, 'auroc': 0.8955296404275995, 'subject': 3}
FT epoch 1/7: loss=0.3694, val_f1=0

{'accuracy': {'mean': 0.8690229921729808,
  'std': 0.19339812410676296,
  'ci95_low': 0.761922680869384,
  'ci95_high': 0.9761233034765777,
  'n': 15},
 'f1': {'mean': 0.7655719739903813,
  'std': 0.34669451057220313,
  'ci95_low': 0.5735789534736293,
  'ci95_high': 0.9575649945071334,
  'n': 15},
 'balanced_accuracy': {'mean': 0.8429020845005766,
  'std': 0.236265792475385,
  'ci95_low': 0.7120624497245127,
  'ci95_high': 0.9737417192766404,
  'n': 15},
 'auroc': {'mean': 0.8869459938103538,
  'std': 0.22841170548471268,
  'ci95_low': 0.760455807435569,
  'ci95_high': 1.0134361801851386,
  'n': 15}}

In [189]:
# 2) SSL + supervised

import json

path = f"/home/leoisidro/CICLOS/X/PFC_II/PFC2/results_optuna/wesad_optuna_{WINDOW_SECONDS}seg.json"

with open(path, "r") as f:
    data = json.load(f)

best_params = data["best_params"]
print(best_params)

res_ssl = run_loso(
X, y, groups,
use_ssl=best_params["use_ssl"],
ssl_epochs=best_params["ssl_epochs"],
ft_epochs=best_params["ft_epochs"],
emb_dim=best_params["emb_dim"],
depth=best_params["depth"],
dropout=best_params["dropout"]
)

summarize("SSL + supervised", res_ssl,f"resultados_SSL_WESAD_{WINDOW_SECONDS}seg.txt")


{'patch_len': 16, 'emb_dim': 128, 'depth': 3, 'dropout': 0.05587873967732661, 'ft_epochs': 6, 'ssl_epochs': 10, 'batch_size': 32, 'use_ssl': True}
SSL epoch 1/10: 2.6840
SSL epoch 2/10: 2.2355
SSL epoch 3/10: 1.9924
SSL epoch 4/10: 1.8732
SSL epoch 5/10: 1.7886
SSL epoch 6/10: 1.6773
SSL epoch 7/10: 1.5977
SSL epoch 8/10: 1.5646
SSL epoch 9/10: 1.4788
SSL epoch 10/10: 1.4323
FT epoch 1/6: loss=0.3347, val_f1=0.8769
FT epoch 2/6: loss=0.2183, val_f1=0.8430
FT epoch 3/6: loss=0.2008, val_f1=0.7869
FT epoch 4/6: loss=0.1487, val_f1=0.8095
FT epoch 5/6: loss=0.1330, val_f1=0.8065
FT epoch 6/6: loss=0.1004, val_f1=0.7669
Fold 1 subject 2 {'accuracy': 0.9552238805970149, 'f1': 0.9302325581395349, 'balanced_accuracy': 0.9680851063829787, 'auroc': 1.0, 'subject': 2}
SSL epoch 1/10: 2.6701
SSL epoch 2/10: 2.2287
SSL epoch 3/10: 1.9986
SSL epoch 4/10: 1.8928
SSL epoch 5/10: 1.8048
SSL epoch 6/10: 1.6914
SSL epoch 7/10: 1.5838
SSL epoch 8/10: 1.5689
SSL epoch 9/10: 1.5266
SSL epoch 10/10: 1.4418


{'accuracy': {'mean': 0.8874413959163328,
  'std': 0.17400981271717725,
  'ci95_low': 0.7910779735824398,
  'ci95_high': 0.9838048182502257,
  'n': 15},
 'f1': {'mean': 0.7990963466946391,
  'std': 0.3252443742662339,
  'ci95_low': 0.6189820157282568,
  'ci95_high': 0.9792106776610214,
  'n': 15},
 'balanced_accuracy': {'mean': 0.8651631900987347,
  'std': 0.21691524708171023,
  'ci95_low': 0.74503953018095,
  'ci95_high': 0.9852868500165193,
  'n': 15},
 'auroc': {'mean': 0.9101233372136733,
  'std': 0.1741599725397176,
  'ci95_low': 0.8136767591417728,
  'ci95_high': 1.0065699152855738,
  'n': 15}}

### 8.2 Selected Hyperparameters for Stress-Predict / Stress-ID


# Run the Optuna hyperparameter search.
print("INICIANDO OPTUNA...")
study_optuna_Stress_ID = run_optuna_search(
    X1, y1, groups_1,
    n_trials=5,
    study_name=f"Stress_ID_optuna_{WINDOW_SECONDS}seg"
)

print("\nMejor F1:", study_optuna_Stress_ID.best_value)
print("Selected hyperparameters:", study_optuna_Stress_ID.best_params)


In [190]:
# 1) Supervised only

import json

path = f"/home/leoisidro/CICLOS/X/PFC_II/PFC2/results_optuna/Stress_ID_optuna_{WINDOW_SECONDS}seg.json"

with open(path, "r") as f:
    data = json.load(f)

best_params = data["best_params"]
print(best_params)

res_sup_1 = run_loso(
X1, y1, groups_1,
use_ssl=False,
ssl_epochs=best_params["ssl_epochs"],
ft_epochs=best_params["ft_epochs"],
emb_dim=best_params["emb_dim"],
depth=best_params["depth"],
dropout=best_params["dropout"]
)

summarize("Supervised only", res_sup_1,f"resultados_Supervisado_StressID_{WINDOW_SECONDS}seg.txt")


{'patch_len': 8, 'emb_dim': 256, 'depth': 4, 'dropout': 0.20059047112039313, 'ft_epochs': 6, 'ssl_epochs': 7, 'batch_size': 16, 'use_ssl': True}
FT epoch 1/6: loss=0.4958, val_f1=0.8340
FT epoch 2/6: loss=0.4597, val_f1=0.8769
FT epoch 3/6: loss=0.4286, val_f1=0.8706
FT epoch 4/6: loss=0.4254, val_f1=0.9004
FT epoch 5/6: loss=0.4173, val_f1=0.8791
FT epoch 6/6: loss=0.4076, val_f1=0.8986
Fold 1 subject 2 {'accuracy': 0.94, 'f1': 0.9538461538461539, 'balanced_accuracy': 0.9210526315789473, 'auroc': 0.9847198641765705, 'subject': 2}
FT epoch 1/6: loss=0.4846, val_f1=0.8413
FT epoch 2/6: loss=0.4459, val_f1=0.8848
FT epoch 3/6: loss=0.4259, val_f1=0.8636
FT epoch 4/6: loss=0.4162, val_f1=0.8837
FT epoch 5/6: loss=0.4256, val_f1=0.8989
FT epoch 6/6: loss=0.4022, val_f1=0.8675
Fold 2 subject 3 {'accuracy': 0.8571428571428571, 'f1': 0.8636363636363636, 'balanced_accuracy': 0.88, 'auroc': 0.9717647058823529, 'subject': 3}
FT epoch 1/6: loss=0.5201, val_f1=0.8803
FT epoch 2/6: loss=0.4473, val

{'accuracy': {'mean': 0.8152653065315262,
  'std': 0.15255266414138094,
  'ci95_low': 0.7611724839194312,
  'ci95_high': 0.8693581291436212,
  'n': 33},
 'f1': {'mean': 0.8292250292600631,
  'std': 0.14442723168878016,
  'ci95_low': 0.7780133597158538,
  'ci95_high': 0.8804366988042724,
  'n': 33},
 'balanced_accuracy': {'mean': 0.8203703312721674,
  'std': 0.15479987739434173,
  'ci95_low': 0.7654806814886288,
  'ci95_high': 0.875259981055706,
  'n': 33},
 'auroc': {'mean': 0.8576534238127733,
  'std': 0.20807113503389305,
  'ci95_low': 0.7838746074188837,
  'ci95_high': 0.9314322402066629,
  'n': 33}}

In [191]:
# 2) SSL + supervised

import json

path = f"/home/leoisidro/CICLOS/X/PFC_II/PFC2/results_optuna/Stress_ID_optuna_{WINDOW_SECONDS}seg.json"

with open(path, "r") as f:
    data = json.load(f)

best_params = data["best_params"]
print(best_params)

res_ssl_1 = run_loso(
X1, y1, groups_1,
use_ssl=best_params["use_ssl"],
ssl_epochs=best_params["ssl_epochs"],
ft_epochs=best_params["ft_epochs"],
emb_dim=best_params["emb_dim"],
depth=best_params["depth"],
dropout=best_params["dropout"]
)
summarize("SSL + supervised", res_ssl_1,f"resultados_SSL_StressID_{WINDOW_SECONDS}seg.txt")


{'patch_len': 8, 'emb_dim': 256, 'depth': 4, 'dropout': 0.20059047112039313, 'ft_epochs': 6, 'ssl_epochs': 7, 'batch_size': 16, 'use_ssl': True}
SSL epoch 1/7: 2.4217
SSL epoch 2/7: 1.8400
SSL epoch 3/7: 1.7863
SSL epoch 4/7: 1.7801
SSL epoch 5/7: 1.7046
SSL epoch 6/7: 1.6648
SSL epoch 7/7: 1.5900
FT epoch 1/6: loss=0.4764, val_f1=0.8815
FT epoch 2/6: loss=0.4213, val_f1=0.8604
FT epoch 3/6: loss=0.4074, val_f1=0.8849
FT epoch 4/6: loss=0.4106, val_f1=0.8905
FT epoch 5/6: loss=0.3940, val_f1=0.8806
FT epoch 6/6: loss=0.3795, val_f1=0.8745
Fold 1 subject 2 {'accuracy': 0.96, 'f1': 0.96875, 'balanced_accuracy': 0.9473684210526316, 'auroc': 0.9779286926994907, 'subject': 2}
SSL epoch 1/7: 2.4127
SSL epoch 2/7: 1.8465
SSL epoch 3/7: 1.7248
SSL epoch 4/7: 1.6262
SSL epoch 5/7: 1.5807
SSL epoch 6/7: 1.5381
SSL epoch 7/7: 1.4694
FT epoch 1/6: loss=0.4480, val_f1=0.8913
FT epoch 2/6: loss=0.3871, val_f1=0.8682
FT epoch 3/6: loss=0.4091, val_f1=0.8873
FT epoch 4/6: loss=0.3791, val_f1=0.8780
FT

{'accuracy': {'mean': 0.8188745917515527,
  'std': 0.1440293745502896,
  'ci95_low': 0.7678039962143984,
  'ci95_high': 0.8699451872887071,
  'n': 33},
 'f1': {'mean': 0.8451502785950838,
  'std': 0.11660803750363738,
  'ci95_low': 0.8038028663861233,
  'ci95_high': 0.8864976908040444,
  'n': 33},
 'balanced_accuracy': {'mean': 0.8156730725099506,
  'std': 0.15690173546069439,
  'ci95_low': 0.7600381362630195,
  'ci95_high': 0.8713080087568816,
  'n': 33},
 'auroc': {'mean': 0.8852773134755237,
  'std': 0.17788347983575353,
  'ci95_low': 0.8222025741746091,
  'ci95_high': 0.9483520527764384,
  'n': 33}}

## 9. Cross-Dataset Transfer Evaluation

This section trains and selects models using the source dataset and evaluates the resulting model on the target dataset. The implementation should be audited together with the saved prediction-level outputs used to reproduce Table 2.


In [192]:
import json
import os

import numpy as np
import torch
from sklearn.model_selection import train_test_split


def load_best_params(study_name):
    path = os.path.join(RESULTS_DIR, f"{study_name}.json")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)["best_params"]


def train_on_source_eval_target(source_name, X_src, y_src, target_name, X_tgt, y_tgt, best_params):
    torch.cuda.empty_cache()
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

    batch_size = int(best_params.get("batch_size", 64))
    patch_len = int(best_params.get("patch_len", 8))

    X_train, X_val, y_train, y_val = train_test_split(
        X_src,
        y_src,
        test_size=0.15,
        random_state=SEED,
        stratify=y_src,
    )

    model = TimePatchTransformer(
        in_ch=3,
        patch_len=patch_len,
        emb_dim=best_params["emb_dim"],
        depth=best_params["depth"],
        heads=4,
        dropout=best_params["dropout"],
        num_classes=2,
    )

    if best_params.get("use_ssl", False):
        ssl_batch_size = min(max(2, batch_size), len(X_train))
        model = pretrain_ssl(
            model,
            X_train.copy(),
            epochs=best_params["ssl_epochs"],
            batch_size=ssl_batch_size,
        )

    ft_batch_size = min(max(2, batch_size), len(X_train))
    model = finetune(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=best_params["ft_epochs"],
        batch_size=ft_batch_size,
    )

    test_batch_size = min(max(2, batch_size), len(X_tgt))
    probs = predict_probs(model, X_tgt, batch_size=test_batch_size)
    metrics = evaluate_probs(y_tgt, probs)

    print(f"{source_name} -> {target_name}")
    for key in ["accuracy", "f1", "balanced_accuracy", "auroc"]:
        print(f"{key}: {metrics[key]:.4f}")

    return metrics


wesad_params = load_best_params(f"wesad_optuna_{WINDOW_SECONDS}seg")
stress_params = load_best_params(f"Stress_ID_optuna_{WINDOW_SECONDS}seg")

res_wesad_to_stress = train_on_source_eval_target(
    "WESAD",
    X,
    y,
    "STRESS-ID",
    X1,
    y1,
    wesad_params,
)
summarize(
    "WESAD->STRESS-ID",
    [res_wesad_to_stress],
    f"resultados_Cruzados_WESAD_a_STRESSID_all_{WINDOW_SECONDS}seg.txt",
    append=False,
)

res_stress_to_wesad = train_on_source_eval_target(
    "STRESS-ID",
    X1,
    y1,
    "WESAD",
    X,
    y,
    stress_params,
)
summarize(
    "STRESS-ID->WESAD",
    [res_stress_to_wesad],
    f"resultados_Cruzados_STRESSID_a_WESAD_all_{WINDOW_SECONDS}seg.txt",
    append=False,
)

print("Cross-dataset training/evaluation completed.")


SSL epoch 1/10: 2.0409
SSL epoch 2/10: 1.6906
SSL epoch 3/10: 1.4794
SSL epoch 4/10: 1.3093
SSL epoch 5/10: 1.2073
SSL epoch 6/10: 1.1594
SSL epoch 7/10: 1.0655
SSL epoch 8/10: 1.0586
SSL epoch 9/10: 1.0559
SSL epoch 10/10: 1.0052
FT epoch 1/6: loss=0.3539, val_f1=0.8478
FT epoch 2/6: loss=0.2019, val_f1=0.9263
FT epoch 3/6: loss=0.1726, val_f1=0.8791
FT epoch 4/6: loss=0.1487, val_f1=0.9091
FT epoch 5/6: loss=0.1322, val_f1=0.9362
FT epoch 6/6: loss=0.1064, val_f1=0.9213
WESAD -> STRESS-ID
accuracy: 0.5860
f1: 0.5077
balanced_accuracy: 0.6355
auroc: 0.8048

WESAD->STRESS-ID
  accuracy              : 0.5860 ± nan  [IC95%: (nan, nan)]  n=1
  f1                    : 0.5077 ± nan  [IC95%: (nan, nan)]  n=1
  balanced_accuracy     : 0.6355 ± nan  [IC95%: (nan, nan)]  n=1
  auroc                 : 0.8048 ± nan  [IC95%: (nan, nan)]  n=1


/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


SSL epoch 1/7: 1.3801
SSL epoch 2/7: 2.1792
SSL epoch 3/7: 1.9820
SSL epoch 4/7: 1.7053
SSL epoch 5/7: 1.7451
SSL epoch 6/7: 1.7519
SSL epoch 7/7: 2.5011
FT epoch 1/6: loss=0.5065, val_f1=0.7752
FT epoch 2/6: loss=0.4624, val_f1=0.7479
FT epoch 3/6: loss=0.4597, val_f1=0.7891
FT epoch 4/6: loss=0.4496, val_f1=0.7773
FT epoch 5/6: loss=0.4505, val_f1=0.7947
FT epoch 6/6: loss=0.4437, val_f1=0.8013
STRESS-ID -> WESAD
accuracy: 0.6000
f1: 0.5819
balanced_accuracy: 0.6983
auroc: 0.8872

STRESS-ID->WESAD
  accuracy              : 0.6000 ± nan  [IC95%: (nan, nan)]  n=1
  f1                    : 0.5819 ± nan  [IC95%: (nan, nan)]  n=1
  balanced_accuracy     : 0.6983 ± nan  [IC95%: (nan, nan)]  n=1
  auroc                 : 0.8872 ± nan  [IC95%: (nan, nan)]  n=1
Cross-dataset training/evaluation completed.


/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


## 10. Combined-Dataset Evaluation

This section evaluates training on pooled data from both datasets using subject-level partitions and evaluates performance on held-out participants.


In [193]:
import json
import os

import numpy as np
import torch
from sklearn.model_selection import train_test_split


def load_best_params(study_name):
    path = os.path.join(RESULTS_DIR, f"{study_name}.json")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)["best_params"]


def resample_window_to_length(window, target_len):
    window = np.asarray(window, dtype=np.float32)
    if window.shape[0] == target_len:
        return window
    xp = np.linspace(0.0, 1.0, window.shape[0])
    x = np.linspace(0.0, 1.0, target_len)
    resized = [np.interp(x, xp, window[:, channel]) for channel in range(window.shape[1])]
    return np.stack(resized, axis=1).astype(np.float32)


def resize_dataset(X, target_len):
    return np.stack([resample_window_to_length(sample, target_len) for sample in X], axis=0)


def train_combined_and_eval(combo_name, X_src_a, y_src_a, groups_a, X_src_b, y_src_b, groups_b, best_params):
    torch.cuda.empty_cache()
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

    common_len = min(X_src_a.shape[1], X_src_b.shape[1])
    X_src_a = resize_dataset(X_src_a, common_len)
    X_src_b = resize_dataset(X_src_b, common_len)

    print(f"--- Leakage-safe subject split: {combo_name} ---")
    print("Splitting Dataset A (WESAD) by participant:")
    tr_a_idx, te_a_idx, tr_subjs_a, te_subjs_a = make_subject_holdout_split(
        groups_a, train_fraction=0.80, random_state=SEED
    )
    print("Splitting Dataset B (STRESS-ID) by participant:")
    tr_b_idx, te_b_idx, tr_subjs_b, te_subjs_b = make_subject_holdout_split(
        groups_b, train_fraction=0.80, random_state=SEED
    )

    X_a_train, y_a_train = X_src_a[tr_a_idx], y_src_a[tr_a_idx]
    X_a_test, y_a_test = X_src_a[te_a_idx], y_src_a[te_a_idx]

    X_b_train, y_b_train = X_src_b[tr_b_idx], y_src_b[tr_b_idx]
    X_b_test, y_b_test = X_src_b[te_b_idx], y_src_b[te_b_idx]

    # Pool development data from both datasets
    X_train_dev = np.concatenate([X_a_train, X_b_train], axis=0)
    y_train_dev = np.concatenate([y_a_train, y_b_train], axis=0)

    # Unique participant tags to prevent ID collision across datasets
    groups_dev = np.concatenate([
        np.array([f"WESAD_{s}" for s in groups_a[tr_a_idx]]),
        np.array([f"STRESS_{s}" for s in groups_b[tr_b_idx]]),
    ])

    # Participant-disjoint validation split from development subjects
    fit_idx, val_idx, fit_subjs, val_subjs = make_subject_holdout_split(
        groups_dev, train_fraction=0.85, random_state=SEED
    )
    X_val_train, y_val_train = X_train_dev[fit_idx], y_train_dev[fit_idx]
    X_val, y_val = X_train_dev[val_idx], y_train_dev[val_idx]

    g_test = np.concatenate([
        np.array([f"WESAD_{s}" for s in groups_a[te_a_idx]]),
        np.array([f"STRESS_{s}" for s in groups_b[te_b_idx]]),
    ])
    assert_disjoint_participants(
        groups_dev[fit_idx],
        val_groups=groups_dev[val_idx],
        test_groups=g_test
    )

    batch_size = int(best_params.get("batch_size", 64))
    patch_len = int(best_params.get("patch_len", 8))

    model = TimePatchTransformer(
        in_ch=3,
        patch_len=patch_len,
        emb_dim=best_params["emb_dim"],
        depth=best_params["depth"],
        heads=4,
        dropout=best_params["dropout"],
        num_classes=2,
    )

    if best_params.get("use_ssl", False):
        ssl_batch_size = min(max(2, batch_size), len(X_val_train))
        model = pretrain_ssl(
            model,
            X_val_train.copy(),
            epochs=best_params["ssl_epochs"],
            batch_size=ssl_batch_size,
        )

    ft_batch_size = min(max(2, batch_size), len(X_val_train))
    model = finetune(
        model,
        X_val_train,
        y_val_train,
        X_val,
        y_val,
        epochs=best_params["ft_epochs"],
        batch_size=ft_batch_size,
    )

    test_a_batch = min(max(2, batch_size), len(X_a_test))
    test_b_batch = min(max(2, batch_size), len(X_b_test))

    metrics_a = evaluate_probs(y_a_test, predict_probs(model, X_a_test, batch_size=test_a_batch))
    metrics_b = evaluate_probs(y_b_test, predict_probs(model, X_b_test, batch_size=test_b_batch))

    print(f"\n{combo_name}")
    print(f"Common window length: {common_len}")
    print("Evaluation on the held-out 20% of WESAD participants")
    for key in ["accuracy", "f1", "balanced_accuracy", "auroc"]:
        print(f"{key}: {metrics_a[key]:.4f}")
    print("Evaluation on the held-out 20% of Stress-Predict / Stress-ID participants")
    for key in ["accuracy", "f1", "balanced_accuracy", "auroc"]:
        print(f"{key}: {metrics_b[key]:.4f}")

    return metrics_a, metrics_b


wesad_params = load_best_params(f"wesad_optuna_{WINDOW_SECONDS}seg")
stress_params = load_best_params(f"Stress_ID_optuna_{WINDOW_SECONDS}seg")

metrics_wesad_params = train_combined_and_eval(
    "Combined Training using WESAD hyperparameters",
    X,
    y,
    groups,
    X1,
    y1,
    groups_1,
    wesad_params,
)
summarize(
    "Combined->WESAD params/test WESAD",
    [metrics_wesad_params[0]],
    f"results_Combined_WESADparams_test_WESAD_{WINDOW_SECONDS}seg.txt",
    append=False,
)
summarize(
    "Combined->WESAD params/test STRESS-ID",
    [metrics_wesad_params[1]],
    f"results_Combined_WESADparams_test_STRESSID_{WINDOW_SECONDS}seg.txt",
    append=False,
)

metrics_stress_params = train_combined_and_eval(
    "Combined Training using STRESS-ID hyperparameters",
    X,
    y,
    groups,
    X1,
    y1,
    groups_1,
    stress_params,
)
summarize(
    "Combinado->STRESS params/test WESAD",
    [metrics_stress_params[0]],
    f"results_Combined_STRESSparams_test_WESAD_{WINDOW_SECONDS}seg.txt",
    append=False,
)
summarize(
    "Combinado->STRESS params/test STRESS-ID",
    [metrics_stress_params[1]],
    f"results_Combined_STRESSparams_test_STRESSID_{WINDOW_SECONDS}seg.txt",
    append=False,
)

print("\Combined experiment 80/20 done.")


--- Leakage-safe subject split: Combined Training using WESAD hyperparameters ---
Splitting Dataset A (WESAD) by participant:
Training participants: 12 | Test participants: 3 | Overlap: 0
Splitting Dataset B (STRESS-ID) by participant:
Training participants: 26 | Test participants: 7 | Overlap: 0
Training participants: 32 | Test participants: 6 | Overlap: 0
SSL epoch 1/10: 1.6045
SSL epoch 2/10: 1.1217
SSL epoch 3/10: 1.0347
SSL epoch 4/10: 0.9644
SSL epoch 5/10: 0.9387
SSL epoch 6/10: 0.9192
SSL epoch 7/10: 0.8570
SSL epoch 8/10: 0.8154
SSL epoch 9/10: 0.7857
SSL epoch 10/10: 0.7765
FT epoch 1/6: loss=0.4467, val_f1=0.7622
FT epoch 2/6: loss=0.3469, val_f1=0.8105
FT epoch 3/6: loss=0.3234, val_f1=0.8562
FT epoch 4/6: loss=0.2978, val_f1=0.8227
FT epoch 5/6: loss=0.2820, val_f1=0.7907
FT epoch 6/6: loss=0.2641, val_f1=0.7960

Combined Training using WESAD hyperparameters
Common window length: 240
Evaluation on the held-out 20% of WESAD participants
accuracy: 0.7799
f1: 0.6102
balanced_

/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


SSL epoch 1/7: 1.4243
SSL epoch 2/7: 1.8124
SSL epoch 3/7: 1.5252
SSL epoch 4/7: 1.2816
SSL epoch 5/7: 1.1314
SSL epoch 6/7: 1.0958
SSL epoch 7/7: 1.0770
FT epoch 1/6: loss=0.4704, val_f1=0.7309
FT epoch 2/6: loss=0.4308, val_f1=0.7379
FT epoch 3/6: loss=0.3914, val_f1=0.7303
FT epoch 4/6: loss=0.3903, val_f1=0.7355
FT epoch 5/6: loss=0.3631, val_f1=0.7568
FT epoch 6/6: loss=0.3513, val_f1=0.7312

Combined Training using STRESS-ID hyperparameters
Common window length: 240
Evaluation on the held-out 20% of WESAD participants
accuracy: 0.8230
f1: 0.7338
balanced_accuracy: 0.8157
auroc: 0.8279
Evaluation on the held-out 20% of Stress-Predict / Stress-ID participants
accuracy: 0.8148
f1: 0.8559
balanced_accuracy: 0.7891
auroc: 0.9342

Combinado->STRESS params/test WESAD
  accuracy              : 0.8230 ± nan  [IC95%: (nan, nan)]  n=1
  f1                    : 0.7338 ± nan  [IC95%: (nan, nan)]  n=1
  balanced_accuracy     : 0.8157 ± nan  [IC95%: (nan, nan)]  n=1
  auroc                 : 0.

/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


## 11. Subject-Level Statistical Analysis

Paired Wilcoxon signed-rank tests compare SSL-initialized and supervised-initialized models using participant-level performance values. Rank-biserial correlation is used as an effect-size measure.


In [194]:
# ============================================================
# Statistical analysis: paired Wilcoxon comparison of SSL and supervised initialization.
# ============================================================
from scipy.stats import wilcoxon as _wilcoxon
from scipy.stats import rankdata as _rankdata
import pandas as pd


def rank_biserial(a, b):
    """Compute rank-biserial correlation as the Wilcoxon effect-size measure.

    Interpretation of the absolute value:
      < 0.10 = trivial | 0.10–0.30 = small
      0.30–0.50 = moderate | > 0.50 = large
    """
    d = np.asarray(a, float) - np.asarray(b, float)
    d = d[d != 0]
    if len(d) == 0:
        return 0.0
    ranks   = _rankdata(np.abs(d))
    r_plus  = float(ranks[d > 0].sum())
    r_minus = float(ranks[d < 0].sum())
    return (r_plus - r_minus) / (r_plus + r_minus)


def wilcoxon_compare(results_a, results_b, dataset_name, metric="f1", alpha=0.05):
    """Perform a paired Wilcoxon signed-rank test.

    Both result lists must contain the same participants and are aligned by participant ID.
    """
    ra = sorted(results_a, key=lambda r: r["subject"])
    rb = sorted(results_b, key=lambda r: r["subject"])
    a_vals = np.array([r[metric] for r in ra], float)
    b_vals = np.array([r[metric] for r in rb], float)

    stat, p = _wilcoxon(a_vals, b_vals, alternative="two-sided")
    rb_c    = rank_biserial(a_vals, b_vals)
    diffs   = a_vals - b_vals

    print(f"\n{'='*60}")
    print(f"  {dataset_name}  |  Metric: {metric.upper()}")
    print(f"{'='*60}")
    print(f"  SSL  (A) — mean: {a_vals.mean():.4f}  std: {a_vals.std(ddof=1):.4f}")
    print(f"  Sup  (B) — mean: {b_vals.mean():.4f}  std: {b_vals.std(ddof=1):.4f}")
    print(f"  Δ A-B    — mean: {diffs.mean():.4f}   mean: {np.mean(diffs):.4f}")
    print(f"  W={stat:.2f}   p={p:.6f}   r_rb={rb_c:.3f}")
    sig = "SIGNIFICANT" if p < alpha else "NOT significant"
    print(f"  {sig}  (α={alpha})")
    return {
        "stat": float(stat), "p": float(p),
        "rank_biserial": float(rb_c),
        "mean_a": float(a_vals.mean()), "mean_b": float(b_vals.mean()),
    }


# Wilcoxon analysis by dataset and metric.
# Requiere: res_sup, res_ssl (WESAD) y res_sup_1, res_ssl_1 (Stress-ID)
# These results are generated by the corresponding experiment cells above.

_metrics_to_test = ["f1", "balanced_accuracy", "auroc"]
wilcoxon_rows = []

print("\n" + "="*70)
print("WILCOXON: SSL vs Supervisado — WESAD + Stress-ID")
print("="*70)

for _metric in _metrics_to_test:
    for _ds_name, _res_a, _res_b in [
        ("WESAD",     res_ssl,   res_sup),
        ("Stress-ID", res_ssl_1, res_sup_1),
    ]:
        _r = wilcoxon_compare(_res_a, _res_b, _ds_name, _metric)
        wilcoxon_rows.append({
            "Dataset":     _ds_name,
            "Metric":     _metric,
            "SSL mean":   round(_r["mean_a"], 4),
            "Sup mean":   round(_r["mean_b"], 4),
            "Δ A-B":       round(_r["mean_a"] - _r["mean_b"], 4),
            "W":           round(_r["stat"], 2),
            "p-value":     round(_r["p"], 6),
            "r_rb":        round(_r["rank_biserial"], 3),
            "Sig (α=0.05)":_r["p"] < 0.05,
        })

wilcoxon_df = pd.DataFrame(wilcoxon_rows)

# Bonferroni correction.
_n_tests    = len(wilcoxon_rows)
_alpha_bonf = 0.05 / _n_tests
wilcoxon_df["Sig (Bonferroni)"] = wilcoxon_df["p-value"] < _alpha_bonf
print(f"\nBonferroni correction: α_corr = 0.05/{_n_tests} = {_alpha_bonf:.4f}")
display(wilcoxon_df.round(4))



WILCOXON: SSL vs Supervisado — WESAD + Stress-ID

  WESAD  |  Metric: F1
  SSL  (A) — mean: 0.7991  std: 0.3252
  Sup  (B) — mean: 0.8182  std: 0.2966
  Δ A-B    — mean: -0.0191   mean: -0.0191
  W=15.00   p=0.373840   r_rb=-0.333
  NOT significant  (α=0.05)

  Stress-ID  |  Metric: F1
  SSL  (A) — mean: 0.8452  std: 0.1166
  Sup  (B) — mean: 0.8292  std: 0.1444
  Δ A-B    — mean: 0.0159   mean: 0.0159
  W=262.00   p=0.970168   r_rb=0.008
  NOT significant  (α=0.05)

  WESAD  |  Metric: BALANCED_ACCURACY
  SSL  (A) — mean: 0.8652  std: 0.2169
  Sup  (B) — mean: 0.8758  std: 0.2082
  Δ A-B    — mean: -0.0106   mean: -0.0106
  W=12.00   p=0.213122   r_rb=-0.467
  NOT significant  (α=0.05)

  Stress-ID  |  Metric: BALANCED_ACCURACY
  SSL  (A) — mean: 0.8157  std: 0.1569
  Sup  (B) — mean: 0.8204  std: 0.1548
  Δ A-B    — mean: -0.0047   mean: -0.0047
  W=246.50   p=0.743487   r_rb=-0.066
  NOT significant  (α=0.05)

  WESAD  |  Metric: AUROC
  SSL  (A) — mean: 0.9101  std: 0.1742
  Sup  

,Dataset,Metric,SSL mean,Sup mean,Δ A-B,W,p-value,r_rb,Sig (α=0.05),Sig (Bonferroni)
0,WESAD,f1,0.7991,0.8182,-0.0191,15.0,0.3738,-0.333,False,False
1,Stress-ID,f1,0.8452,0.8292,0.0159,262.0,0.9702,0.008,False,False
2,WESAD,balanced_accuracy,0.8652,0.8758,-0.0106,12.0,0.2131,-0.467,False,False
3,Stress-ID,balanced_accuracy,0.8157,0.8204,-0.0047,246.5,0.7435,-0.066,False,False
4,WESAD,auroc,0.9101,0.9076,0.0025,7.0,0.4631,0.333,False,False
5,Stress-ID,auroc,0.8853,0.8577,0.0276,185.0,0.3286,0.204,False,False


## 12. Partition and Index Documentation



In [195]:
# ============================================================
# Documentation of participant-level 80/20 partitions.
# ============================================================
import pandas as pd


def display_partition_log(partition_log, dataset_name):
    """Display a summary table of participants and window indices by partition.

    Parameters
    ----------
    partition_log : dictionaries returned by run_split8020_per_subject.
    dataset_name : dataset identifier.

    Returns
    -------
    pd.DataFrame with one row per participant.
    """
    rows = []
    for p in partition_log:
        rows.append({
            "Participant":              p["subject"],
            "N train windows (participant)":   p["n_train_subject"],
            "N test windows (participant)":    p["n_test_subject"],
            "N train windows (total)": p["n_train_total"],
            "Train cls 0":         p["train_class_dist"].get(0, 0),
            "Train cls 1":         p["train_class_dist"].get(1, 0),
            "Test cls 0":          p["test_class_dist"].get(0, 0),
            "Test cls 1":          p["test_class_dist"].get(1, 0),
            "Participant IDs in training":    str(p["train_subjects_used"]),
        })

    df = pd.DataFrame(rows).set_index("Participant").sort_index()

    print(f"\n{'='*70}")
    print(f"80/20 partitions — {dataset_name}")
    print(f"{'='*70}")
    print(f"  Seed     : {partition_log[0]['seed']}")
    print(f"  test_frac: {partition_log[0]['test_frac']}")
    print(f"  N participants: {len(partition_log)}\n")

    try:
        display(df.style.set_caption(
            f"Window distribution by participant — {dataset_name}"
        ))
    except Exception:
        print(df.to_string())

    return df


# Summary tables for WESAD and Stress-Predict / Stress-ID (if available).
if 'plog_wesad' in globals() and 'plog_sp' in globals():
    df_parts_wesad = display_partition_log(plog_wesad, "WESAD")
    df_parts_sp    = display_partition_log(plog_sp,    "Stress-ID")

    # Detailed example for one participant.
    print("\n" + "─"*60)
    print("Detailed example — WESAD")
    print("─"*60)
    _ej = plog_wesad[0]
    print(f"Participant {_ej['subject']}:")
    print(f"  Participants en training : {_ej['train_subjects_used']}")
    print(f"  Windows en TEST  ({len(_ej['test_window_indices'])} total) — first 20 indices:")
    print(f"    {_ej['test_window_indices'][:20]}")
    print(f"  Training windows for the participant ({len(_ej['train_window_indices'])} total) — first 20 indices:")
    print(f"    {_ej['train_window_indices'][:20]}")

    print("\n" + "─"*60)
    print("Detailed example — Stress-ID")
    print("─"*60)
    _ej2 = plog_sp[0]
    print(f"Participant {_ej2['subject']}:")
    print(f"  Participants en training : {_ej2['train_subjects_used']}")
    print(f"  Windows en TEST  ({len(_ej2['test_window_indices'])} total) — first 20 indices:")
    print(f"    {_ej2['test_window_indices'][:20]}")
    print(f"  Training windows for the participant ({len(_ej2['train_window_indices'])} total) — first 20 indices:")
    print(f"    {_ej2['train_window_indices'][:20]}")
else:
    print('Partition logs (plog_wesad, plog_sp) are not available in current session.')


Partition logs (plog_wesad, plog_sp) are not available in current session.


## Public Repository Safety and Reproducibility Notes

- Replace generic paths such as `/path/to/project/` and `/path/to/project/` with paths appropriate to the execution environment.
- Do not commit raw participant data to the repository.
- Preserve only pseudonymous dataset participant identifiers needed to reproduce subject-level splits.
- Run participant-overlap checks for every experimental partition.
- Estimate normalization and imputation parameters from training data only.
- Keep target-dataset labels out of training, validation, model selection, and threshold calibration in cross-dataset experiments.
- Publish exact split definitions, selected hyperparameters, software versions, and random seeds as reproducibility metadata.


## Leakage-Safe Subject Partitioning for Experiment 4

For subject-independent evaluation, the 80/20 split must refer to **participants**, not windows within each participant. The function below partitions unique participant identifiers first and only then obtains their corresponding windows. This guarantees that no participant contributes data to both training and held-out testing.


In [196]:
def make_subject_holdout_split(groups, train_fraction=0.80, random_state=42):
    """Create a participant-disjoint train/test split.

    Parameters
    ----------
    groups : array-like
        Participant identifier for every window.
    train_fraction : float
        Fraction of unique participants assigned to model development.
    random_state : int
        Seed controlling the participant split.

    Returns
    -------
    train_idx, test_idx : np.ndarray
        Window indices belonging exclusively to training or test participants.
    train_subjects, test_subjects : np.ndarray
        Disjoint participant identifiers.
    """
    groups = np.asarray(groups)
    subjects = np.unique(groups)

    train_subjects, test_subjects = train_test_split(
        subjects,
        train_size=train_fraction,
        random_state=random_state,
        shuffle=True,
    )

    train_idx = np.flatnonzero(np.isin(groups, train_subjects))
    test_idx = np.flatnonzero(np.isin(groups, test_subjects))

    overlap = set(train_subjects).intersection(set(test_subjects))
    if overlap:
        raise RuntimeError(f"Participant leakage detected: {sorted(overlap)}")

    print(f"Training participants: {len(train_subjects)}")
    print(f"Test participants: {len(test_subjects)}")
    print("Participant overlap: 0")
    return train_idx, test_idx, train_subjects, test_subjects
